# KuchoLM training

Colab向けの最小学習ノートブックです。JSONLの `source` / `target` / `style` を読み込み、SentencePiece tokenizer と小型Encoder-Decoder Transformerを学習します。

想定JSONL:
```json
{"style":"OJOU","source":"今日は暑いですね。","target":"本日は暑うございますわね。"}
```

In [ ]:
!pip -q install sentencepiece

In [ ]:
from pathlib import Path
import json
import math
import random
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

DATA_PATH = Path('/content/kucholm.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 8000
MAX_LEN = 128
D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
DIM_FEEDFORWARD = 1024
DROPOUT = 0.1
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 3e-4
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. データ読込
`style` を `<STYLE>` タグとして入力先頭に付けます。

In [ ]:
rows = []
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        style = str(item['style']).strip().upper()
        source = str(item['source']).strip()
        target = str(item['target']).strip()
        if source and target and style:
            rows.append((f'<{style}> {source}', target))

random.shuffle(rows)
print('rows:', len(rows))
print(rows[0] if rows else 'no data')

## 2. SentencePiece tokenizer

In [ ]:
corpus_path = WORK_DIR / 'spm_corpus.txt'
with corpus_path.open('w', encoding='utf-8') as f:
    for source, target in rows:
        f.write(source + '\n')
        f.write(target + '\n')

model_prefix = str(WORK_DIR / 'kucholm_spm')
spm.SentencePieceTrainer.train(
    input=str(corpus_path),
    model_prefix=model_prefix,
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=0.9995,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

sp = spm.SentencePieceProcessor(model_file=model_prefix + '.model')
print('vocab:', sp.vocab_size())

## 3. Dataset

In [ ]:
PAD_ID = sp.pad_id()
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()

def encode(text):
    ids = sp.encode(text, out_type=int)[:MAX_LEN - 2]
    return [BOS_ID, *ids, EOS_ID]

class KuchoDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        source, target = self.items[index]
        return torch.tensor(encode(source)), torch.tensor(encode(target))

def collate(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    return srcs, tgts

split = max(1, int(len(rows) * 0.95))
train_rows = rows[:split]
val_rows = rows[split:] or rows[:min(32, len(rows))]

train_loader = DataLoader(KuchoDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(KuchoDataset(val_rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

## 4. Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(sp.vocab_size(), D_MODEL, padding_idx=PAD_ID)
        self.positional = PositionalEncoding(D_MODEL)
        self.transformer = nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=NUM_ENCODER_LAYERS,
            num_decoder_layers=NUM_DECODER_LAYERS,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            batch_first=True,
        )
        self.lm_head = nn.Linear(D_MODEL, sp.vocab_size(), bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, src, tgt):
        src_key_padding_mask = src.eq(PAD_ID)
        tgt_key_padding_mask = tgt.eq(PAD_ID)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)
        src_emb = self.positional(self.embedding(src) * math.sqrt(D_MODEL))
        tgt_emb = self.positional(self.embedding(tgt) * math.sqrt(D_MODEL))
        hidden = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.lm_head(hidden)

model = KuchoTransformer().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'{params / 1_000_000:.2f}M parameters')

## 5. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for src, tgt in train_loader:
        src = src.to(device)
        tgt = tgt.to(device)
        decoder_input = tgt[:, :-1]
        labels = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        logits = model(src, decoder_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src = src.to(device)
            tgt = tgt.to(device)
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            val_loss += loss.item()

    train_avg = train_loss / max(1, len(train_loader))
    val_avg = val_loss / max(1, len(val_loader))
    print(f'epoch={epoch} train={train_avg:.4f} val={val_avg:.4f}')


## 6. Save

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size': sp.vocab_size(),
        'max_len': MAX_LEN,
        'd_model': D_MODEL,
        'nhead': NHEAD,
        'num_encoder_layers': NUM_ENCODER_LAYERS,
        'num_decoder_layers': NUM_DECODER_LAYERS,
        'dim_feedforward': DIM_FEEDFORWARD,
    },
}
torch.save(checkpoint, WORK_DIR / 'kucholm.pt')
print(WORK_DIR / 'kucholm.pt')
print(WORK_DIR / 'kucholm_spm.model')

## 7. Greedy inference

In [ ]:
@torch.no_grad()
def convert_style(text, style='OJOU', max_new_tokens=96):
    model.eval()
    src = torch.tensor([encode(f'<{style.upper()}> {text}')], device=device)
    generated = torch.tensor([[BOS_ID]], device=device)

    for _ in range(max_new_tokens):
        logits = model(src, generated)
        next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break

    ids = generated[0].tolist()
    ids = [token_id for token_id in ids if token_id not in {PAD_ID, BOS_ID, EOS_ID}]
    return sp.decode(ids)

print(convert_style('今日はとても暑いですね。', 'OJOU'))